# Developing data simulation

The objective of this notebook is helping to develop step by step the simulation from `R` into `python`

The first objective will be the creation of a simulation possibility based only on continuous features. No categorical will be considered.

In [1]:
import torch

In [2]:
means = torch.tensor([3.5,-3.5], dtype=torch.float64)
covs = torch.tensor([[1,-0.5],[-0.5,1]], dtype=torch.float64)
mvn = torch.distributions.MultivariateNormal(means, covariance_matrix=covs)

In a 2 dimensional multivariate distribution, we have a vector $X$ of two RV
$$
X = \begin{pmatrix} X_1 \\ X_2 \end{pmatrix}
$$
Mean and VCOV matrix are given by
$$
\mu = \begin{pmatrix} 3.5 \\ -3.5 \end{pmatrix}, \; 
\Sigma = \begin{pmatrix}
1 & -\frac{1}{2} \\
-\frac{1}{2} & 1
\end{pmatrix}
$$

This means when sampling we will only get positive numbers for $X_1$ realizations and negative numbers for $X_2$ realizations

In [3]:
sample_2d = mvn.sample((3,))
print(sample_2d.shape)
sample_2d

torch.Size([3, 2])


tensor([[ 4.0739, -2.5579],
        [ 3.5667, -2.2065],
        [ 4.5279, -4.0313]], dtype=torch.float64)

So in a sample size of `(n,)`, the shape will be `[n, k]` with `k` the number of random variables composing $X$.

With $x^{(r)}_j$ the $j$-th sampled realization of $X_r$, the output looks like
$$
\texttt{sample} = \begin{pmatrix}
    x^{(1)}_1 & \cdots & x^{(k)}_1 \\
    \vdots & \ddots & \vdots \\
    x^{(1)}_n & \cdots & x^{(k)}_n
\end{pmatrix} \in \mathbb{R}^{n \times k}
$$

## Notes on the R-Algorithm

### Calls:

First call is made for the init_population, by

```r
res <- genCreditData(
  #################################### DIMENSIONALITY
  n                = init_sample,  # - sample size = 100
  bad_ratio        = bad_ratio,    # - BAD ratio (if all D = 0) = 0.7
  k_con            = num_feats,    # - no. continuous features = 2
  k_cat            = 0,            # - no. categorical features
  k_bin            = 0,            # - no. binary features
  k_noise          = num_noise,    # - no. white-noise features = 0
  #################################### CONTINUOUS FEATURES
  con_nonlinear    = 0.0,       # - share of nonlinear transformations
  con_mean_bad_dif = mean_dif,  # - mean difference between classes = c(2, 1)
  con_var_bad_dif  = var_dif,   # - share of var/covar difference between classes = 0.5
  con_noise_var    = noise_var, # - variance of noise = 0
  covars           = covars,    # - variance-covariance matrices = list(matrix(c(1,  0.2,  0.2, 1), nrow = 2), matrix(c(1, -0.2, -0.2, 1), nrow = 2))
  #################################### MIXTURE OF GAUSSIANS
  mixture      = mixture,       # - mixture of two Gaussians = FALSE
  mix_mean_dif = mix_mean_dif,  # - mean difference between components  = 5
  mix_var_dif  = mix_var_dif,   # - share of var/covar difference between components = 0
  #################################### OTHER PARAMETERS
  seed             = seed,   # - random seed
  verbose          = F,      # - displaying feedback
  encode_factors   = F)      # - encoding of categorical features
```

`matrix(c(1,  0.2,  0.2, 1)` liefert
$$
\begin{pmatrix}
    1 & 0.2 \\
    0.2 & 1
\end{pmatrix}
$$

The next call is done this way:
```r
new_applicants <- genCreditData(n = sample_size, replicate = res, seed = seed + g)$data
```

Which is a quick way to replicate the arguments of the last call, based on this object (all names are set as objects - which are "`names`" in R)
```r
list(n                = n,
     k_cat            = k_cat,
     k_bin            = k_bin,
     k_noise          = k_noise,
     bad_ratio        = bad_ratio,
     con_mean_bad_dif = con_mean_bad_dif,
     con_var_bad_dif  = con_var_bad_dif,
     con_nonlinear    = con_nonlinear,
     con_noise_var    = con_noise_var,
     mixture          = mixture,
     mix_mean_dif     = mix_mean_dif,
     mix_var_dif      = mix_var_dif,
     cat_levels       = cat_levels,
     cat_var_share    = cat_var_share,
     cat_nonlinear    = cat_nonlinear,
     cat_noise_var    = cat_noise_var,
     bin_prob         = bin_prob,
     bin_mean_bad_dif = bin_mean_bad_dif,
     bin_bad_ratio    = bin_bad_ratio,
     bin_mean_con_dif = bin_mean_con_dif,
     bin_var_bad_dif  = bin_var_bad_dif,
     bin_noise_var    = bin_noise_var,
     encode_factors   = encode_factors,
     verbose          = verbose,
     seed             = seed)
```

however n and seed are not "taken over" - the rest they are

### Relevant parts of the algorithm

1. Set `combo_bad_ratio <- 0.7` and `combo_count <- n`
2. Compute "good" and "bad" sample sizes, as $n \cdot 0.7$ und $n \cdot (1-0.7)$ correspondingly. Ties are solved by 50/50 chance of doing good = n - bad or bad = n - good.
3. Set `mu_1 = c(0,0)` and therefore `mu_2 = c(1,2)=con_mean_bad_dif` (see line 249)

In [239]:
from typing import List, Optional, Tuple, Union
import torch
def random_vcov_matrix(
        k: int,
        generator: Optional[torch.Generator] = None,
        var_range: Tuple[float, float] = (0.0, 1.0),
        prefer_normal_base_sampling: bool = True,
        device: torch.device = torch.get_default_device(),
        dtype: torch.dtype = torch.get_default_dtype(),
        eps: float = 1e-6
    ) -> torch.Tensor:
    """Generate a random positive definite covariance matrix.

    The covariance matrix is produced by:

    1. Generating a base sampling of a matrix :math:`A` (normal or uniform).
    2. Creating a correlation matrix via cosine similarity between the row vectors
       of the base sampling:

       .. math::

          C = (c_{i,j}) = \\left( \\frac{\\langle A_i, A_j \\rangle}{\\|A_i\\| \\|A_j\\|} \\right)

    3. Sampling a variance vector uniformly in ``var_range``, which is used to
       rescale the correlation matrix.
    4. Ensuring positive definiteness via addition of a small diagonal
       perturbation ``eps``.
    5. Make sure exact symmetry, so that rounding point instability does not
       lead to unsymmetric results. 

    Args:
        k (int): Dimension of the covariance matrix.
        generator (torch.Generator, optional): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances. Defaults to (0.0, 1.0).
        prefer_normal_base_sampling (bool, optional): If True, use normal distribution for base sampling.
            If False, use uniform distribution. Defaults to True.
        device (torch.device, optional): Device on which to allocate the tensor.
            Defaults to ``torch.get_default_device()``.
        dtype (torch.dtype, optional): Data type of the returned tensor.
            Defaults to ``torch.get_default_dtype()``.
        eps (float, optional): Small positive value added to the diagonal to ensure positive definiteness.
            Defaults to 1e-6.

    Returns:
        torch.Tensor: A symmetric, positive definite covariance matrix of shape ``(k, k)``.

    Raises:
        ValueError: If ``var_range`` is not a valid (min, max) tuple.

    Example:
        >>> g = torch.Generator().manual_seed(42)
        >>> cov = random_vcov_matrix(4, generator=g)
        >>> cov.shape
        torch.Size([4, 4])
    """

    # Step 1: Generate base sampling
    if prefer_normal_base_sampling:
        A = torch.randn((k, k), generator=generator, dtype=dtype, device=device) # random normal matrix, sparser correlations for high k
    else:
        A = 2*torch.rand((k,k), generator=generator, dtype = dtype, device=device) - 1 # random uniform matrix, correlations closer to 0, the higher k

    # Step 2: Define correlation matrix from base sampling
    Q = A @ A.T # Make sure of symmetry while using full randomness
    D = torch.sqrt(torch.diag(Q)) # Help vector for normalization
    corr_mat = Q / torch.outer(D, D) # corr_mat[i, j] = cosine_similarity(A[i], A[j]), so range [-1, 1] guaranteed

    # Step 3: Rescale corr_mat with sampled variances
    ## Variance sampling from uniform distribution
    variances = torch.rand(k, generator=generator, dtype = dtype, device=device) * (var_range[1] - var_range[0]) + var_range[0]
    ## Rescaling via outer prouct of standard deviations
    stds = torch.sqrt(variances)
    norm_factors_pearson_corr = torch.outer(stds, stds) # guaranteed to be symmetric, denominators of pearson correlation
    vcov = corr_mat * norm_factors_pearson_corr

    # Step 4: Avoid semi positive definitness of the matrix
    vcov = vcov + eps * torch.eye(k, device=device, dtype=dtype)

    # Step 5: Ensure **exact** symmetry without compromising randomness
    i, j = torch.tril_indices(k, k, offset=-1)
    vcov[i, j] = vcov[j, i]

    
    return vcov

def eigen_decomp_proj_to_pd(
    mat: torch.Tensor,
    eps: float = 1e-6,
    ensure_symmetry: bool = False
) -> torch.Tensor:
    """Project a matrix onto the positive definite (PD) cone via eigen-decomposition.

    The procedure ensures the output is symmetric and positive semidefinite by:
    
    1. Optionally symmetrizing the input matrix.
    2. Performing eigen-decomposition.
    3. Clipping eigenvalues below ``eps`` to enforce non-negativity.
    4. Reconstructing the matrix from clipped eigenvalues and eigenvectors.
    5. Symmetrizing the result again to avoid numerical drift.

    Args:
        mat (torch.Tensor): Input square matrix of shape ``(k, k)``.
        eps (float, optional): Minimum eigenvalue threshold to enforce positive definiteness.
            Defaults to ``1e-6``.
        ensure_symmetry (bool, optional): If True, symmetrize the input before decomposition.
            Defaults to False.

    Returns:
        torch.Tensor: Symmetric positive semidefinite matrix of shape ``(k, k)``.

    Example:
        >>> M = torch.tensor([[1.0, 2.0], [2.0, -3.0]])
        >>> M_psd = eigen_decomp_proj_to_pd(M)
        >>> torch.linalg.eigvalsh(M_psd)
        tensor([1.0133e-06, 1.8284e+00])
    """
    # Ensure symmetry
    if ensure_symmetry:
        mat = (mat + mat.T) / 2
    
    # Eigen-decomposition
    eigvals, eigvecs = torch.linalg.eigh(mat)
    
    # Clip eigenvalues to non-negative
    eigvals_clipped = torch.clamp(eigvals, min=eps)
    
    # Reconstruct
    mat_psd = eigvecs @ torch.diag(eigvals_clipped) @ eigvecs.T
    
    # Ensure symmetry again
    return (mat_psd + mat_psd.T) / 2

def generate_sigma_bad_and_good(
    k: int,
    proportion_var_dif: float,
    generator: torch.Generator,
    var_range: Tuple[float, float] = (0.0, 1.0),
    eps: float = 1e-6,
    device: torch.device = torch.device("cpu"),
    dtype: torch.dtype = torch.float64
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Generate a pair of covariance matrices: one 'good' baseline and one 'bad' perturbed version.

    The construction proceeds as follows:

    1. Generate two baseline covariance matrices using ``random_vcov_matrix``.
    2. Sample a random mask over the upper-triangular entries (including diagonal).
    3. Copy selected entries from the 'good' matrix into the 'bad' matrix, leaving
       others perturbed.
    4. Reflect the upper-triangular entries to the lower-triangular part to ensure symmetry.
    5. Project the 'bad' matrix onto the positive definite cone using
       :func:`eigen_decomp_proj_to_pd`.

    Args:
        k (int): Dimension of the covariance matrices.
        proportion_var_dif (float): Probability of keeping an entry different between
            the 'bad' and 'good' matrices.
        generator (torch.Generator): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances.
            Defaults to (0.0, 1.0).
        eps (float, optional): Small diagonal perturbation to ensure positive definiteness.
            Defaults to ``1e-6``.
        device (torch.device, optional): Device for tensor allocation. Defaults to CPU.
        dtype (torch.dtype, optional): Data type of the returned tensors. Defaults to ``torch.float64``.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]:
            - ``sigma_bad``: Perturbed covariance matrix of shape ``(k, k)``, projected to PSD.
            - ``sigma_good``: Baseline covariance matrix of shape ``(k, k)``.

    Example:
        >>> g = torch.Generator().manual_seed(123)
        >>> sigma_bad, sigma_good = generate_sigma_bad_and_good(3, 0.5, generator=g)
        >>> sigma_bad.shape, sigma_good.shape
        (torch.Size([3, 3]), torch.Size([3, 3]))
    """
    # Step 1: Generate baseline matrices
    sigma_bad = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)
    sigma_good = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)

    # Step 2: Random mask for off-diagonal entries
    count_possible_changes = (k**2 + k) // 2 #Count diagonal entries + upper triangle
    index_change_vars = ~torch.bernoulli(torch.full((count_possible_changes,), proportion_var_dif, device=device), generator=generator).bool()

    triu_indices = torch.triu_indices(k, k, offset=0)
    indices_to_copy_sigma_bad = (triu_indices[0][index_change_vars], triu_indices[1][index_change_vars])

    sigma_good[indices_to_copy_sigma_bad] = sigma_bad[indices_to_copy_sigma_bad]
    i, j = torch.tril_indices(k, k, offset=-1)
    sigma_good[i, j] = sigma_good[j, i] # ensure symmetry
    
    sigma_good = eigen_decomp_proj_to_pd(sigma_good, eps=eps)

    return sigma_bad, sigma_good

def mvn_random_sample(
        mean : torch.Tensor, 
        cov_chol_decomp : Optional[torch.Tensor],
        n : int, 
        rng : Optional[torch.Generator] = None, 
        args_checks : bool = True
    ):
    """Generate n-vectors sampled of a multivariate normal (MVN) distribution with parameters
    mean and cov. Based on the implementation of (r)sample from 
    torch.distributions.MultivariateNormal according to torch version 2.9.1. It uses
    cholesky-decomposition method.

    Args:
        mean (torch.Tensor): Location parameter of a MVN. Shape ``(k,)`` or ``(b, k)`` or ``(1,5)``.
        cov_chol_decomp (torch.Tensor): Variance-Covariance matrix of MVN after cholesky decomposition. 
            Shape ``(k,k)``` or ``(b, k, k)`` or ``(1, k, k)``, ``cov.dim()==mean.dim()+1`` should hold.
        n (int): Count of vectors to be sampled (per batch).
        rng (Optional[torch.Generator]): If passed, sampling is done using this
            generator.
        args_checks (bool): If true, it will be checked whether the shapes of mean and
            cov are as expected, whether symmetry (w. r. t. to the last wo dims for each beach)
            is given within the range of ``symmetry_rtol_atol`` for ``cov`` and type checks
            are done for ``n`` and ``rng``.`
        symmetry_rtol_atol (Tuple[float,float]): Corresponds to the (rtol, a_tol) parameters
            of ``torch.allclose``, passed as ``*args``, so ordering is important. Ignored if
            ``not args_checks``.
    Returns:
        torch.Tensor:
            A tensor of shape ``(n, k)`` or ``(n, b, k)`` containing the ``n`` sampled vectors (for each batch).

    Example:
        >>> count_covariates = 5
        >>> device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        >>> torch.set_default_dtype(torch.float32)
        >>> rng = torch.Generator(device)
        >>> mu = torch.zeros(count_covariates)
        >>> sigma_bad, _ = generate_sigma_bad_and_good(k = count_covariates, proportion_var_dif=1.0, generator = rng, device=device, dtype= torch.get_default_dtype())
        >>> sample = mvn_random_sample(mean=mu, cov=sigma_bad, n=100, rng=rng)
        >>> sample.shape
        torch.Size([100, 5])
    """
    if args_checks:
        #shape checks
        assert (mean.dim() in [1, 2, 3]) and (cov_chol_decomp.dim()==mean.dim()+1), "mean must be a single vector (rank 1 tensor) and cov a matrix (rank 2 tensor)"
        assert (mean.size(-1) == cov_chol_decomp.size(-1)) and (cov_chol_decomp.size(-1) == cov_chol_decomp.size(-2)), "mean must have shape [k] and cov shape [k, k]"
        #ensure n is an int
        n = int(n)
        assert isinstance(rng, torch.Generator) or rng is None, "rng needs to be None or a rng"

    shape = torch.Size([n]) + mean.shape
    
    eps = torch.empty(shape, dtype=mean.dtype, device = mean.device).normal_(generator=rng)
    cov_chol_decomp = torch.linalg.cholesky(cov) # occurs over the last dimension

    deviations = torch.matmul(cov_chol_decomp, eps.unsqueeze(-1)).squeeze(-1) # apply decomp to each sampled vector


    return mean + deviations


# Extending CreditDataSample

Notes and experiments on how to keep tabs of the feature ids

In [1]:
import torch
from typing import Optional
class CreditDataSample:
    def __init__(
        self,
        features_rejects: torch.Tensor,
        features_accepts: torch.Tensor,
        default_flag_accepts: torch.Tensor,
        ids_rejects : Optional[torch.Tensor] = None,
        retrieve_only_accepted: bool = True,
        seed: Optional[int] = None,
    ):
        self.features_unlabeled = features_rejects
        self.features_labeled = features_accepts
        self.labels = default_flag_accepts

        self.retrieve_only_accepted = retrieve_only_accepted

        # RNG is device-specific, so we create it on the same device as the data
        device = features_rejects.device
        self.rng = torch.Generator(device=device)
        if seed is not None:
            self.rng.manual_seed(seed)

        # Logic to keep track of observations which were labeled
        rej_batch_shape = features_rejects.shape[:-1]
        if ids_rejects is None:
            count_rej_obs = torch.prod(torch.tensor(rej_batch_shape))
            self._rej_ids = torch.arange(count_rej_obs).reshape(*rej_batch_shape)
        else:
            assert ids_rejects.shape == rej_batch_shape, "ids_rejects has the wrong shape. Should be features_rejects.shape[:-1]"
            self._rej_ids = ids_rejects

        self.mask_inferred_rejs = torch.full(rej_batch_shape, fill_value=False) # to get ids of rejected where inference was made

        self.acc_batch_shape = default_flag_accepts.shape
        self.mask_inferred_lbls = torch.tensor(False).expand(self.acc_batch_shape) # to get only inferred labels
        self._ids_inferred = torch.tensor(torch.nan).expand(self.acc_batch_shape)

B, N_r, N_a, C = 32, 500, 200, 5

data = sample = CreditDataSample(
    features_rejects=torch.randn(B, N_r, C),
    features_accepts=torch.randn(B, N_a, C),
    default_flag_accepts=torch.randint(0,2, (B, N_a), dtype=torch.get_default_dtype())
)

In [2]:
nan_val = -1
inferred_labels = torch.distributions.Categorical(torch.tensor([0.8,0.1,0.1])).sample(data.features_unlabeled.shape[:-1]) - 1
mask_inferred_rej_lbls = inferred_labels != -1

self = sample
inplace : bool = False
safety_checks : bool = True

In [ ]:
from typing import Dict, Tuple, Union
def _padded_gather(
        gather_from : torch.Tensor, 
        mask : torch.Tensor,
        nan_value : Union[int, float] = float('nan')
) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    mask_int = mask.to(torch.int32)
    precalc_tensors = {
        "valid_counts" : mask.sum(-1),
        "valid_pos" : torch.where(# [B, N]
            mask,
            mask_int.cumsum(-1) - 1,
            -1
        )
    }
    return _padded_gather_with_precalc(gather_from, mask, nan_value=nan_value, **precalc_tensors), precalc_tensors

def _padded_gather_with_precalc(
        gather_from : torch.Tensor, 
        mask : torch.Tensor,
        valid_pos : torch.Tensor,
        valid_counts : torch.Tensor,
        nan_value : Union[int, float] = float('nan')
    ) -> torch.Tensor:
    assert gather_from.shape[:-1] == mask.shape

    *super_batch_shape, N, K = gather_from.shape # [..., N, K]
    no_super_batches = len(super_batch_shape) == 0 # -> gather_from.shape=[N, K]
    if no_super_batches:
        gathered = gather_from[mask] # [n, K]
        return gathered

    gather_from = gather_from.flatten(0,-3) # [B, N, K]
    B = gather_from.size(0)
    mask = mask.flatten(0,-2) # [B, N]

    max_valid = valid_counts.max()
    gathered = gather_from.new_full((B, max_valid, K), nan_value)

    batch_idx = torch.arange(B, device=gather_from.device).unsqueeze(-1).expand(B, N)
    gathered[batch_idx[mask], valid_pos[mask]] = gather_from[mask]

    return gathered.reshape(*super_batch_shape, max_valid, K) # [..., N, K]

if True:
    if True:
        if safety_checks:
            if mask_inferred_rej_lbls.dtype != torch.bool:
                raise ValueError("mask_infered_rej_lbls should be of type bool")
            if not (mask_inferred_rej_lbls.shape == inferred_labels.shape == self.features_unlabeled.shape[:-1]):
                raise ValueError("mask_inferred_rej_lbls and inferred_labels should have the same shape as self.features_unlabeled.shape[:-1]")
            
            inferred_labels_with_vals_as_saved_labels = torch.isin(inferred_labels[mask_inferred_rej_lbls].unique(), self.labels).all()
            if not inferred_labels_with_vals_as_saved_labels:
                raise ValueError("Tensor inferred_labels[mask_inferred_rej_lbls] should contain only values like in self.labels")
        
        with torch.no_grad():
            inferred_feats_rej, precalcs_inferred = _padded_gather(self.features_unlabeled, mask_inferred_rej_lbls)
            ids_inferred_feats_rej = _padded_gather_with_precalc(self._rej_ids.unsqueeze(-1).to(float), 
                                                                 mask_inferred_rej_lbls, 
                                                                 **precalcs_inferred).squeeze(-1)
            inferred_labels = _padded_gather_with_precalc(
                inferred_labels.unsqueeze(-1).to(self.labels.dtype), 
                mask_inferred_rej_lbls,
                nan_value=(torch.nan if torch.is_floating_point(self.labels) else -1),
                **precalcs_inferred
            ).squeeze(-1)
            
            lbld_features = torch.cat([self.features_labeled, inferred_feats_rej], dim=-2)
            lbls = torch.cat([self.labels, inferred_labels], dim=-1)
            inferred_ids = torch.cat([self._inferred_ids, ids_inferred_feats_rej], dim=-1)


            non_inferred_feats_rej, precals_noninf = _padded_gather(self.features_unlabeled, ~mask_inferred_rej_lbls)
            ids_non_inferred_feats_rej = _padded_gather_with_precalc(self._rej_ids.unsqueeze(-1).to(float),
                                                                    ~mask_inferred_rej_lbls, 
                                                                    **precals_noninf).squeeze(-1)

                                                                    

        

In [22]:
self.labels.dtype
torch.is_floating_point(self.labels)

True

In [170]:
def vectorized_splitting(X, mask):
    # X: [B, N, K]
    # mask: [B, N] (bool)

    B, N, K = X.shape
    mask_int = mask.to(torch.int32)

    # ---- inferred rejects ----

    # Compute positions 0..n_i-1 for inferred rows
    inferred_pos = torch.where(# [B, N]
        mask,
        mask_int.cumsum(-1) - 1,
        -1
    )

    inferred_counts = mask.sum(-1)  # [B]
    max_inferred = inferred_counts.max()

    # Create padded output
    inferred_feats = X.new_full((B, max_inferred, K), float('nan'))

    # Broadcast batch indices
    batch_idx = torch.arange(B, device=X.device).unsqueeze(-1).expand(B, N)

    # Valid positions mask
    valid_inferred = mask

    # Scatter rows into padded tensor
    inferred_feats[batch_idx[valid_inferred], inferred_pos[valid_inferred]] = X[valid_inferred]


    # ---- non-inferred rejects ----

    non_mask_int = (~mask).to(torch.int32)

    non_pos = torch.where(
        mask,
        -1,
        non_mask_int.cumsum(-1) - 1
    )
    

    non_counts = N - inferred_counts
    max_non = non_counts.max()

    non_inferred_feats = X.new_full((B, max_non, K), float('nan'))

    valid_non = ~mask

    non_inferred_feats[batch_idx[valid_non], non_pos[valid_non]] = X[valid_non]

    return inferred_feats, non_inferred_feats


inferred_feats_vec, non_inferred_feats_vec = vectorized_splitting(reshaped_unlabeled_features, reshaped_mask_inferred)
((inferred_feats_vec.reshape(*leading_bdim_shape, -1, K) == inferred_feats_rej) | inferred_feats_vec.isnan()).all()

tensor(True)

In [160]:
import time
import pandas as pd

times = []

for t in range(1000):
    inferred_feats_rej = reshaped_unlabeled_features[b][reshaped_mask_inferred[b]] # [n_i, K]
    n_i = inferred_feats_rej.size(0)
    begin_cat = time.time()
    lower_padding_non_inferred = torch.full((max_inferred_count-n_i, K), torch.nan, dtype=feats_dtype, device=feats_device)
    inferred_feats_rej_cat = torch.cat([inferred_feats_rej, lower_padding_non_inferred],dim=0)
    end_cat = time.time()
    inferred_feats_rej_pad = torch.nn.functional.pad(
        inferred_feats_rej, 
        pad=(0,0,0, max_inferred_count-n_i),
        mode='constant',
        value=torch.nan
    )
    end_pad = time.time()

    if t == 0:
        assert ((inferred_feats_rej_pad == inferred_feats_rej_cat) | inferred_feats_rej_cat.isnan()).all()

    times.append({"cat" : end_cat - begin_cat, "pad" : end_pad - end_cat})

times = pd.DataFrame(times)
times["pad/cat"] = times["pad"] / times["cat"]
times.describe()

,cat,pad,pad/cat
count,1000.000000,1000.000000,1000.000000
mean,0.000019,0.000015,0.859090
std,0.000016,0.000008,0.350097
min,0.000016,0.000013,0.185072
25%,0.000016,0.000014,0.821918
50%,0.000017,0.000014,0.842857
75%,0.000017,0.000015,0.867647
max,0.000313,0.000143,8.585714


In [133]:
torch.isin(inferred_labels[mask_inferred_rej_lbls].unique(), self.labels).all()

tensor(True)

In [135]:
inferred_labels[mask_inferred_rej_lbls].unique().dtype, self.labels.dtype

(torch.int64, torch.float32)